# Semantic Chunking

Semantic chunking: Splitting text into contextually meaningful chunks using semantic similarity, preserving related information to improve retrieval, search accuracy, and understanding.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np 

In [5]:
# intialaize the model

model = SentenceTransformer("all-MiniLM-L6-v2")

# sample text

text = """" 
         LangChain is an open-source framework designed to simplify the
         development of applications powered by large language models (LLMs). It
         provides reusable components that help developers build intelligent
         paris is the capital of france 
         football is most popular in france
"""

#step 1 : split in into sentences

sentences = [s for s in text.split("\n") if s.strip()]


#step 2 : embed each sentences 

embeddings = model.encode(sentences)

#step 3 : threshhold parameter

threshold = 0.7
chunks = []
current_chunk = [sentences[0]]

# step 4 : semantic grouping based on threshold

for i in range(1,len(sentences)):
    sim = cosine_similarity(
        [embeddings[i-1]],
        [embeddings[i]]
    )[0][0]

    if sim >= threshold:
        current_chunk.append(sentences[i])

    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]


# append the last chunks

chunks.append(" ".join(current_chunk))



#output of the chunks


print("\n semantic chunks: ")
for idx , chunk in enumerate(chunks):
    print(f"chunks , {idx+1}:\n{chunk}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3380.81it/s]



 semantic chunks: 
chunks , 1:
" 
chunks , 2:
         LangChain is an open-source framework designed to simplify the
chunks , 3:
         development of applications powered by large language models (LLMs). It
chunks , 4:
         provides reusable components that help developers build intelligent
chunks , 5:
         paris is the capital of france 
chunks , 6:
         football is most popular in france


### RAG Pipeline Modular Coding

In [16]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Document
from langchain_core.documents import Document

# Vector Store
from langchain_community.vectorstores import FAISS

# OpenAI Embeddings
from langchain_openai import OpenAIEmbeddings

# Chat Model
from langchain.chat_models import init_chat_model

# LCEL Runnables
from langchain_core.runnables import RunnableLambda, RunnableParallel , RunnableMap

# Prompt
from langchain_core.prompts import PromptTemplate

# Output Parser
from langchain_core.output_parsers import StrOutputParser



import os 
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [8]:
### Custom Semantic Chunker With Threshold

class ThresholdSematicChunker:
    def __init__(self,model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model=SentenceTransformer(model_name)
        self.threshold=threshold 

    def split(self, text: str):
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]

        chunks.append(". ".join(current_chunk) + ".")
        return chunks
    
    def split_documents(self,docs):
        result=[]
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))

        return result

    

In [9]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [10]:
### Chunking
chunker=ThresholdSematicChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9781.14it/s]


[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [11]:
### VectorStore
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
embedding=OpenAIEmbeddings()
vectorstore=FAISS.from_documents(chunks,embedding)
retriever=vectorstore.as_retriever()



In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

In [14]:
## Prompt Template

# --- 5. Prompt Template ---
template = """Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [17]:
rag_chain=(
    RunnableMap(
        {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],  
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

# --- 8. Run Query ---
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)

LangChain is used as a framework for building applications with large language models (LLMs). It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.


## Semantic chunker with langchain

In [22]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

In [23]:
## Load the documents
loader=TextLoader("langchain_intro.txt")
docs=loader.load()

## Initialize embedding model
embedding=OpenAIEmbeddings()

## Create the semantic chunker
chunker=SemanticChunker(embedding)

## Split the documents
chunks=chunker.split_documents(docs)

## Result

for i,chunk in enumerate(chunks):
    print(f"\n chunk {i+1}:\n{chunk.page_content}")


 chunk 1:
LangChain Introduction

LangChain is an open-source framework for building applications powered by large language models. It provides components for prompt management, document loading, text splitting, embeddings, vector stores, retrieval, memory, and agents. Developers use LangChain to create production-ready AI applications. Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) combines information retrieval with large language models. Documents are loaded, split into chunks, embedded into vectors, and stored inside a vector database.

 chunk 2:
When a user asks a question, relevant chunks are retrieved and sent to the LLM to generate an accurate answer. Machine Learning

Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed. Common algorithms include Linear Regression, Decision Trees, Random Forest, Support Vector Machines, and Neural Networks. Deep Learning

Deep